# Fine-tune LLMs for CTF/Coding - Self-Contained Pipeline

Complete pipeline: **data scraping → synthesis → training → export**

**Setup:**
- Runtime → Change runtime type → **T4 GPU**
- Run cells top-to-bottom (1-10)
- No external files needed - everything is built in

## Section 1: Install Dependencies

In [ ]:
# ============================================================
# 1.1 Detect environment (Colab vs local)
# ============================================================
import os, re, sys, json, time, shutil, tempfile
from pathlib import Path

IS_COLAB = "COLAB_" in "".join(os.environ.keys())
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")

# ============================================================
# 1.2 Install all required packages
# ============================================================
if IS_COLAB:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    print(f"PyTorch: {torch.__version__}, xformers: {xformers}")
    
    # Data scraping deps
    !pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer requests gitpython pyyaml tqdm
    
    # Unsloth + training deps (--no-deps avoids version conflicts)
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
    
    torch._dynamo.config.recompile_limit = 64

# ============================================================
# 1.3 Import all libraries
# ============================================================
import requests, yaml
from git import Repo
from datasets import load_dataset
from tqdm import tqdm
import torch

# ============================================================
# 1.4 Verify GPU
# ============================================================
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: Configuration

All tunable parameters in one place. Edit this cell to adjust model, LoRA rank, training epochs, etc.

In [ ]:
# ============================================================
# 2.1 Model selection
# ============================================================
# Options:
#   - "unsloth/Qwen3.5-4B"   (4B, fastest, fits T4 with 4-bit QLoRA)
#   - "unsloth/Qwen3.5-9B"   (9B, better quality, fits T4 with 4-bit QLoRA)
#   - "unsloth/gemma-4-E4B-it" (4B, multimodal, 16-bit LoRA)
MODEL_NAME = "unsloth/Qwen3.5-4B"

# ============================================================
# 2.2 LoRA hyperparameters
# ============================================================
LORA_R = 16            # Rank: 8, 16, 32, 64 (higher = more capacity)
LORA_ALPHA = 16        # Usually = r, or 2*r for faster training
MAX_SEQ_LENGTH = 4096  # 2048 if OOM, 8192 if lots of VRAM

# ============================================================
# 2.3 Training hyperparameters
# ============================================================
BATCH_SIZE = 1         # T4 limit
GRAD_ACCUM = 4         # Effective batch = 1*4 = 4
NUM_EPOCHS = 3         # 1 for quick test, 3 for full
LEARNING_RATE = 2e-4   # 2e-4 standard, 1e-5 for longer runs
WARMUP_STEPS = 10

# ============================================================
# 2.4 Data limits
# ============================================================
MAX_PER_REPO = 200        # Cap per GitHub repo
MAX_HF_SAMPLES = 3000     # Cap per HuggingFace dataset
MAX_DOC_SECTIONS = 20     # Cap per doc source

# ============================================================
# 2.5 Output
# ============================================================
OUTPUT_DIR = "/content/outputs/qwen4b-ctf"

# ============================================================
# 2.6 Show all config
# ============================================================
CONFIG = {
    "model_name": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_per_repo": MAX_PER_REPO,
    "max_hf_samples": MAX_HF_SAMPLES,
    "output_dir": OUTPUT_DIR,
}
print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Section 3: Scrape CTF Writeups from GitHub

Clones 10 GitHub repos and extracts Q&A pairs from writeup markdown files.

In [ ]:
# ============================================================
# 3.1 Define GitHub repos to scrape
# ============================================================
CTF_WRITEUP_REPOS = [
    # picoCTF (2 repos)
    {"url": "https://github.com/Cajac/picoCTF-Writeups", "name": "picoctf-cajac", "category": "picoctf"},
    {"url": "https://github.com/vivian-dai/PicoCTF2021-Writeup", "name": "picoctf-2021", "category": "picoctf"},
    # CryptoHack (2 repos)
    {"url": "https://github.com/DarkCodeOrg/CryptoHack", "name": "cryptohack-darkcode", "category": "cryptohack"},
    {"url": "https://github.com/AyushSingh-c/Cryptohack", "name": "cryptohack-ayush", "category": "cryptohack"},
    # pwnCollege (3 repos)
    {"url": "https://github.com/H3xKatana/pwncollege-writeups", "name": "pwncollege-h3x", "category": "pwncollege"},
    {"url": "https://github.com/prettyb0iisam/pwncollege-writeups", "name": "pwncollege-prettyb0i", "category": "pwncollege"},
    {"url": "https://github.com/id-none/pwncollege_writeup", "name": "pwncollege-idnone", "category": "pwncollege"},
    # Multi-category (3 repos)
    {"url": "https://github.com/Adamkadaban/CTFs", "name": "ctfs-adamkadaban", "category": "multi"},
    {"url": "https://github.com/Cryptogenic/Exploit-Writeups", "name": "exploit-writeups", "category": "pwn"},
    {"url": "https://github.com/ffffffff0x/1earn", "name": "0x1earn", "category": "multi"},
]
print(f"Configured {len(CTF_WRITEUP_REPOS)} repos to scrape")

In [ ]:
# ============================================================
# 3.2 Helper functions for repo cloning and markdown extraction
# ============================================================
def clone_repo(url, dest):
    """Clone git repo (shallow), cleaning up if already exists."""
    dest_path = Path(dest)
    if dest_path.exists():
        shutil.rmtree(dest_path)
    try:
        Repo.clone_from(url, dest, depth=1)
        return dest_path.exists()
    except Exception as e:
        print(f"  Clone failed: {e}")
        return False

def find_solution_boundary(content):
    """Find line index where '## Solution' or similar markers begin."""
    markers = [
        r'^##\s+(?:Solution|Writeup|Exploit|Answer|Flag)',
        r'^###\s+(?:Solution|Writeup|Exploit|Answer|Flag)',
        r'\*\*(?:Solution|Exploit|Answer|Flag)\*\*'
    ]
    lines = content.split('\n')
    for i, line in enumerate(lines):
        for marker in markers:
            if re.match(marker, line, re.IGNORECASE):
                return i
    return len(lines)

def extract_code_blocks(content):
    """Extract markdown code blocks (```python, ```c, etc.)."""
    pattern = r'```(\w+)?\n(.*?)```'
    return [{"lang": lang or "text", "code": code.strip()}
            for lang, code in re.findall(pattern, content, re.DOTALL)
            if len(code.strip()) > 10]

print("Helper functions defined")

In [ ]:
# ============================================================
# 3.3 Extract Q&A pairs from markdown writeups
# ============================================================
def extract_writeup(md_file, category):
    """Convert one markdown file into a Q&A example."""
    try:
        content = md_file.read_text(errors='ignore')
        if len(content) < 100:
            return None
        # Skip root READMEs (usually index pages)
        if md_file.name.lower() == "readme.md":
            return None

        challenge_name = md_file.stem.replace("-", " ").replace("_", " ").title()
        boundary = find_solution_boundary(content)
        description = re.sub(r'```.*?```', '',
                                '\n'.join(content.split('\n')[:boundary]),
                                flags=re.DOTALL)
        description = re.sub(r'\n{3,}', '\n\n', description).strip()[:2000]
        solution = re.sub(r'\n{3,}', '\n\n',
                            '\n'.join(content.split('\n')[boundary:])).strip()[:3000]

        if not description or len(description) < 50:
            return None

        # Collect code from markdown + sibling files
        code_blocks = extract_code_blocks(content)
        for cf in list(md_file.parent.glob("*.py"))[:2] + list(md_file.parent.glob("*.c"))[:2]:
            try:
                code = cf.read_text(errors='ignore')
                if len(code) > 10:
                    code_blocks.append({"lang": cf.suffix[1:] or "text", "code": code})
            except:
                pass

        # Build output: solution text + code blocks
        if code_blocks:
            output = f"## Solution\n\n{solution}\n\n" + "\n\n".join(
                f"```{b['lang']}\n{b['code'][:1500]}\n```" for b in code_blocks[:3])
        else:
            output = solution or f"## Solution\n\n{description}"
        if len(output) < 50:
            return None

        instruction = "Write an exploit/solution for this challenge" if code_blocks else "Explain how to solve this challenge step by step"
        return {
            "instruction": instruction,
            "input": f"Challenge: {challenge_name}\nCategory: {category}\n\n{description}",
            "output": output
        }
    except Exception:
        return None

# ============================================================
# 3.4 Clone all repos and extract writeups
# ============================================================
all_writeups = []
with tempfile.TemporaryDirectory() as tmpdir:
    for repo in tqdm(CTF_WRITEUP_REPOS, desc="Repos"):
        dest = f"{tmpdir}/{repo['name']}"
        if clone_repo(repo["url"], dest):
            repo_path = Path(dest)
            count = 0
            for md in list(repo_path.rglob("*.md")) + list(repo_path.rglob("*.MD")):
                if md.name.lower() == "readme.md":
                    continue
                ex = extract_writeup(md, repo["category"])
                if ex:
                    all_writeups.append(ex)
                    count += 1
                    if count >= MAX_PER_REPO:
                        break
            print(f"  {repo['name']}: {count} examples")
        else:
            print(f"  {repo['name']}: SKIPPED")

print(f"\nExtracted {len(all_writeups)} writeups total")

## Section 4: Download HuggingFace Datasets

Pulls pre-curated datasets: CTFtime writeups, OpenCodeReasoning (competitive programming), cybersecurity Q&A, etc.

In [ ]:
# ============================================================
# 4.1 Define HF datasets to download
# ============================================================
HF_DATASETS = [
    ("Jacqkues/ctf_webserver_v0.1", 340),                   # Web CTF
    ("kyleavery/picoctf", 120),                            # picoCTF (test split)
    ("justinwangx/CTFtime", MAX_HF_SAMPLES),               # CTF writeups
    ("nvidia/OpenCodeReasoning", MAX_HF_SAMPLES),          # Competitive programming
    ("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1", MAX_HF_SAMPLES),  # Cybersecurity Q&A
    ("ayshajavd/code-security-vulnerability-dataset", MAX_HF_SAMPLES),     # Vulnerability detection
]
print(f"Configured {len(HF_DATASETS)} HF datasets")

In [ ]:
# ============================================================
# 4.2 Download and convert HF datasets to Alpaca format
# ============================================================
def load_hf_with_fallback(name):
    """Try train, then test, then default split."""
    for split in ["train", "test"]:
        try:
            return load_dataset(name, split=split)
        except Exception:
            continue
    return load_dataset(name)

def extract_qa(item):
    """Extract question/answer from various HF dataset schemas."""
    # Messages format (chat)
    msgs = item.get("messages")
    if msgs and len(msgs) >= 2:
        q = msgs[0].get("content", "")
        a = msgs[-1].get("content", "")
        if q and a:
            return q, a
    # Standard fields
    q = item.get("question") or item.get("problem") or item.get("description") or ""
    a = item.get("answer") or item.get("solution") or item.get("flag") or ""
    if q and a:
        return q, a
    # text_chunk format (CTFtime)
    text = item.get("text_chunk", "")
    if text:
        return "Explain this CTF writeup and provide the solution", text
    return None, None

hf_examples = []
for name, max_samples in tqdm(HF_DATASETS, desc="HF datasets"):
    try:
        ds = load_hf_with_fallback(name)
        count = 0
        for item in tqdm(ds, desc=f"  {name.split('/')[-1]}", leave=False):
            if count >= max_samples:
                break
            q, a = extract_qa(item)
            if q and a:
                hf_examples.append({"instruction": q, "input": "", "output": a})
                count += 1
        print(f"  {name}: {count} examples")
    except Exception as e:
        print(f"  Failed {name}: {e}")

print(f"\nDownloaded {len(hf_examples)} HF examples total")

## Section 5: Scrape Security Tool Documentation

Pulls READMEs and reference docs for pwntools, angr, z3, pwndbg, ROPgadget, Ropper, sympy, gmpy2, sagemath.

In [ ]:
# ============================================================
# 5.1 Define documentation sources
# ============================================================
DOC_SOURCES = [
    {"url": "https://raw.githubusercontent.com/Gallopsled/pwntools-write-ups/master/README.md", "name": "pwntools"},
    {"url": "https://raw.githubusercontent.com/angr/angr/master/README.md", "name": "angr"},
    {"url": "https://raw.githubusercontent.com/Z3Prover/z3/master/README.md", "name": "z3"},
    {"url": "https://raw.githubusercontent.com/pwndbg/pwndbg/dev/FEATURES.md", "name": "pwndbg"},
    {"url": "https://raw.githubusercontent.com/JonathanSalwan/ROPgadget/master/README.md", "name": "ropgadget"},
    {"url": "https://raw.githubusercontent.com/sashs/Ropper/master/README.md", "name": "ropper"},
    {"url": "https://raw.githubusercontent.com/david942j/one_gadget/master/README.md", "name": "one_gadget"},
    {"url": "https://raw.githubusercontent.com/sympy/sympy/master/README.rst", "name": "sympy"},
    {"url": "https://raw.githubusercontent.com/gmpy2/gmpy2/master/README.md", "name": "gmpy2"},
    {"url": "https://raw.githubusercontent.com/sagemath/sage/master/README.md", "name": "sagemath"},
]
print(f"Configured {len(DOC_SOURCES)} documentation sources")

In [ ]:
# ============================================================
# 5.2 Scrape and parse documentation
# ============================================================
def scrape_doc(url, name):
    """Download a doc page and split into Q&A sections."""
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        content = r.text
        if len(content) < 100:
            return []
        # Split by ## headers
        sections = re.split(r'\n(?=## )', content)
        examples = []
        for sec in sections[:MAX_DOC_SECTIONS]:
            if len(sec) < 50:
                continue
            title_match = re.match(r'##\s+(.+)', sec)
            title = title_match.group(1) if title_match else "Documentation"
            examples.append({
                "instruction": f"Explain how to use {name} for: {title}",
                "input": sec[:1500],
                "output": f"Based on {name} docs:\n\n{sec[:1500]}"
            })
        return examples
    except Exception as e:
        print(f"  Failed {name}: {e}")
        return []

doc_examples = []
for src in tqdm(DOC_SOURCES, desc="Docs"):
    examples = scrape_doc(src["url"], src["name"])
    doc_examples.extend(examples)
    print(f"  {src['name']}: {len(examples)} sections")

print(f"\nScraped {len(doc_examples)} doc examples total")

## Section 6: Convert to ChatML and Merge

Combines all sources (writeups + HF + docs) into ChatML format with auto-selected system prompts (CTF vs coding).

In [ ]:
# ============================================================
# 6.1 Define system prompts
# ============================================================
SYSTEM_CTF = """You are an expert CTF player. You specialize in:
- Binary exploitation: buffer overflows, ROP, heap, format strings
- Reverse engineering: binary analysis, deobfuscation
- Web: SQL injection, XSS, SSRF, deserialization
- Cryptography: cryptanalysis, key recovery
- Forensics: memory/network analysis, steganography

Analyze systematically, identify the vulnerability, provide step-by-step solution with code."""

SYSTEM_CODING = """You are an expert competitive programmer. You specialize in:
- Algorithm design and analysis
- Data structure optimization
- Code optimization and debugging
- Security-aware coding

Understand requirements, identify optimal approach, write clean efficient code."""

CTF_KEYWORDS = ["pwn", "rev", "web", "crypto", "ctf", "exploit", "vuln", "shellcode"]

def is_ctf_content(text):
    return any(k in text.lower() for k in CTF_KEYWORDS)

print("System prompts defined")

In [ ]:
# ============================================================
# 6.2 Convert Alpaca → ChatML format
# ============================================================
all_raw = all_writeups + hf_examples + doc_examples
print(f"Total raw examples: {len(all_raw)}")

chatml_data = []
for ex in tqdm(all_raw, desc="Converting to ChatML"):
    instr = ex.get("instruction", "")
    inp = ex.get("input", "")
    out = ex.get("output", "")
    if not instr or not out:
        continue
    user_content = f"{instr}\n\n{inp}" if inp else instr
    sys_prompt = SYSTEM_CTF if is_ctf_content(instr) or is_ctf_content(out) else SYSTEM_CODING
    chatml_data.append({
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_content[:3000]},
            {"role": "assistant", "content": out[:3000]}
        ]
    })

print(f"\nGenerated {len(chatml_data)} ChatML examples")

# ============================================================
# 6.3 Save merged dataset to disk
# ============================================================
os.makedirs("/content/data/merged", exist_ok=True)
with open("/content/data/merged/train.jsonl", "w") as f:
    for ex in chatml_data:
        f.write(json.dumps(ex) + "\n")
print(f"Saved to /content/data/merged/train.jsonl")

## Section 7: Load Model and Configure LoRA

Loads the model with 4-bit QLoRA, applies chat template, configures LoRA adapters.

In [ ]:
# ============================================================
# 7.1 Load base model with 4-bit QLoRA
# ============================================================
from unsloth import FastLanguageModel, get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ============================================================
# 7.2 Apply chatml template
# ============================================================
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
print("Chat template: chatml")

# ============================================================
# 7.3 Configure LoRA adapters
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)
print(f"LoRA configured (r={LORA_R}, alpha={LORA_ALPHA})")

In [ ]:
# ============================================================
# 7.4 Load dataset and apply chat template
# ============================================================
dataset = load_dataset("json", data_files={"train": "/content/data/merged/train.jsonl"}, split="train")
print(f"Dataset: {len(dataset)} examples")

def format_chatml(examples):
    """Apply chatml template to all examples."""
    return {"text": [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["messages"]
    ]}

dataset = dataset.map(format_chatml, batched=True)
print("Chat template applied to all examples")
print(f"\nSample formatted text:\n{dataset[0]['text'][:300]}...")

## Section 8: Train

In [ ]:
# ============================================================
# 8.1 Create SFTTrainer
# ============================================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=float(LEARNING_RATE),
        logging_steps=10,
        output_dir="/content/outputs",
        optim="adamw_8bit",
        seed=3407,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataset_text_field="text",
        dataset_num_proc=1,
        report_to="none",
    ),
)
print("Trainer created")
print(f"  Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total steps: ~{len(dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}")

In [ ]:
# ============================================================
# 8.2 Run training
# ============================================================
print("Starting training...")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {len(dataset)} examples")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print()
trainer.train()
print("\nTraining complete!")

## Section 9: Save and Download

In [ ]:
# ============================================================
# 9.1 Save model in 3 formats
# ============================================================
os.makedirs(f"{OUTPUT_DIR}/lora", exist_ok=True)

# LoRA adapter (for continued training)
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print("LoRA saved")

# GGUF (for llama.cpp / Ollama / LM Studio)
model.save_pretrained_gguf(f"{OUTPUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF saved (q4_k_m)")

# Merged 16-bit (for HuggingFace / vLLM)
model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer, save_method="merged_16bit")
print("Merged model saved (16-bit)")

In [ ]:
# ============================================================
# 9.2 Zip and download to local machine
# ============================================================
from google.colab import files

shutil.make_archive("/content/qwen4b-ctf-output", "zip", "/content/outputs")
print("Created qwen4b-ctf-output.zip")
files.download("/content/qwen4b-ctf-output.zip")
print("Download started!")

## Section 10: (Optional) Test the Trained Model

In [ ]:
# ============================================================
# 10.1 Test inference with a CTF-style prompt
# ============================================================
from transformers import TextStreamer

messages = [
    {"role": "system", "content": SYSTEM_CTF},
    {"role": "user", "content": "Explain how a buffer overflow exploit works."},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)
print("\nTest complete")

In [ ]:
# ============================================================
# 10.2 Test inference with a competitive programming prompt
# ============================================================
messages = [
    {"role": "system", "content": SYSTEM_CODING},
    {"role": "user", "content": "Write a Python function to find the longest palindromic substring in O(n) using Manacher's algorithm."},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
)
print("\nTest complete")